# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyanshu-Technologies/flyrank-ML-track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

For Lane 2, the unit of analysis is one content page's performance observation for one client on one report date. I will use the warehouse's daily content-performance data and work with a mid-panel month such as March 2026 for development and verification.

The decision supported by this data is which content pages should be prioritized for human review. I will keep the final month, June 2026, out of development because it should be treated as a sealed outcome/test period.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features
For the initial review-priority model, I will use at most five observable fields:

1. `gsc_impressions` — indicates the amount of search visibility observed for the page.
2. `gsc_clicks` — indicates the observed search traffic generated by the page.
3. `gsc_avg_position` — indicates the page's observed average search position.
4. `ga4_sessions` — indicates observed site sessions associated with the page.
5. `content_age_days` — indicates how long the content has existed and provides an observable freshness/context signal.

### Label / proxy
`trend_direction` is used as the current proxy label, where `down` indicates that the page is showing a declining trend. This is a prioritization signal, not proof that a refresh will improve performance.

### Context
`client_hash_id`, `content_hash_id`, and `report_date` provide grouping, identification, and time context for the observations. They are useful for analysis and splitting, but are not model features.

### Excluded
I will deliberately exclude `trend_pct` and other fields directly derived from the same trend calculation used to define the proxy label. Including label-derived information would create leakage and make model performance look artificially strong.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import duckdb
from getpass import getpass

# Enter your Hugging Face READ token when prompted.
# Do NOT paste the token directly into this notebook cell.
HF_TOKEN = getpass("Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN '" + HF_TOKEN.replace("'", "''") + "')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected to the FlyRank warehouse.")


Hugging Face READ token:  ········


DuckDB connected to the FlyRank warehouse.


In [ ]:
# Query 1: Verify the grain for March 2026

query_1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_keys
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE month = '2026-03';
"""

con.sql(query_1)

In [ ]:
# Query 2: Verify March 2026 row count and date range

query_2 = f"""
SELECT
    COUNT(*) AS march_row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE month = '2026-03';
"""

con.sql(query_2)

In [ ]:
# Query 3: Verify GSC data availability for March 2026

query_3 = f"""
SELECT
    COUNT(*) AS gsc_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE;
"""

con.sql(query_3)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This warehouse supports observed performance analysis and review prioritization, but it cannot by itself establish that a refresh caused a page's performance to improve.

The history is also unbalanced across clients and content, so observations are not necessarily equally available for every page. Some early observations may contain Google Search Console data without corresponding GA4 data, so availability must be checked before using analytics-based signals.

The 90-day and 30-day windows can overlap with each other, so these windows should not automatically be treated as independent observations.

Finally, the current proxy label describes an observed trend rather than a future outcome. A stronger version of this work would define the target using performance after the decision point and evaluate it on a genuinely future period.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.